# 03_01 Spam and the baseline: what does "good" have to beat?

Before you train anything, you find out what doing nothing scores. By the end of this notebook you will
have counted a corpus two ways and got two different answers, measured a spam filter that never flags a
single message, and found out why its accuracy is the most misleading number in this lab.

**How this notebook works.** The same rhythm as every notebook in this course:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-03-teaching-a-machine-what-spam-looks-like", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'nltk': 'nltk',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib',
           'joblib': 'joblib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import csv
import json
import os
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn import metrics
from spamtools import load_sms, split
from nlpcheck import ask, guess, reveal, check_03_01

SMS = "data/sms_spam/SMSSpamCollection"
print(open(SMS).read()[:400])

## 1. Recall

From Labs 01 and 02. Answer from memory.

**r1.** What does NLTK's English stop list do to the word "not"?
(a) nothing, it is kept, (b) removes it, (c) turns it into "no"

**r2.** In TF-IDF, a word that appears in every document gets
(a) the lowest weight, (b) the highest weight, (c) a weight of exactly 1

**r3.** What does a bag of words throw away? (a) the counts, (b) the vocabulary, (c) the order

In [ ]:
ask("r1", "")
ask("r2", "")
ask("r3", "")

## 2. How many messages are there?

The file is the **SMS Spam Collection**: one text message per line, a label (`ham` for a real message,
`spam`), a tab, then the text. The book's Chapter 3 read it with pandas and reported 5,572 messages.
First count the lines yourself:

In [ ]:
lines = open(SMS).read().splitlines()
print(len(lines), "lines")

Now predict how many rows pandas makes of it with the book's call,
`pd.read_csv(SMS, sep="\t", names=["label", "message"])`.

In [ ]:
guess("pandas_rows", None)   # a number

In [ ]:
book = pd.read_csv(SMS, sep="\t", names=["label", "message"])
reveal("pandas_rows", len(book))

Two messages fewer than there are lines, and no error. Some messages open with a double quote and never
close it. pandas reads a quote as the start of a quoted field, and a quoted field may contain newlines, so
it carries on to the next quote it finds, lines later, and glues everything in between into one message.
Find the evidence: one message whose text contains the next two lines of the file, labels and tabs
included.

In [ ]:
glued = book[book["message"].str.contains("\n")]
print(len(glued), "message contains a line break")
m = glued["message"].iloc[0]
end = m.index("\nham\t", m.index("\nham\t") + 1) + 5
print(repr(m[:end]), "...")

`quoting=csv.QUOTE_NONE` tells pandas a quote is just a character. `spamtools.load_sms()` reads the file
that way, and every notebook in this lab uses it:

In [ ]:
df = load_sms()
print(len(df), "messages")
print(df["label"].value_counts())

## 3. The filter that flags nothing

About one message in seven and a half is spam. Now split the data the way every notebook and the
checkpoint will: 70 percent to train on, 30 percent held back to test on, **stratified** so that both parts
have the same share of spam. `random_state=42` makes the split the same every time.

Then predict the test **accuracy** (the fraction of messages labelled correctly) of a "model" that ignores
the text and always answers `ham`.

In [ ]:
X_train, X_test, y_train, y_test = split(df)
print(len(X_train), "to train,", len(X_test), "to test;",
      (y_train == "spam").mean().round(3), "and", (y_test == "spam").mean().round(3), "spam")
guess("dummy_accuracy", None)   # a fraction, like 0.5

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = dummy.predict(X_test)
acc = metrics.accuracy_score(y_test, pred)
recall = metrics.recall_score(y_test, pred, pos_label="spam")
reveal("dummy_accuracy", round(acc, 3))
print("spam caught:", recall)
print(metrics.confusion_matrix(y_test, pred, labels=["ham", "spam"]))

87 percent accurate, and it has caught none of the 224 spam messages in the test set. The confusion
matrix says it plainly: the top row is the real ham (all 1,449 predicted ham), the bottom row the real spam
(all 224 predicted ham too). **Accuracy rewards agreeing with the majority**, so on imbalanced data it is
the wrong headline number. From here on, every model is judged on how much spam it catches (**recall**)
and how often its flags are right (**precision**).

## 4. But wait: is the test set really unseen?

In [ ]:
seen = X_test.isin(set(X_train)).sum()
print(seen, "of", len(X_test), "test messages also appear, word for word, in the training set")
print(X_test[X_test.isin(set(X_train))].value_counts().head(5))

The corpus has 403 duplicate messages ("Sorry, I'll call later" is a common text), and 178 test messages
have an identical twin in training. A model gets those right by memory, so scores on this split are a
little flattering. The book's split had the same property, and so does most published work on this corpus;
we keep the split so your numbers can be compared, and say so. On your own data, remove duplicates
**before** splitting.

## 5. Your turn: save the baseline

Fill in the four values from what you measured above, then save and check.

In [ ]:
baseline = {
    "n_messages": None,          # YOUR CODE HERE: how many messages load_sms() read
    "spam_share": None,          # YOUR CODE HERE: the fraction of all messages that are spam
    "dummy_accuracy": None,      # YOUR CODE HERE: the dummy's test accuracy
    "dummy_spam_recall": None,   # YOUR CODE HERE: the dummy's test recall on spam
}
os.makedirs("out", exist_ok=True)
json.dump(baseline, open("out/03_01_baseline.json", "w"), indent=1)
check_03_01()

## 6. Exit ticket

Explain it back: a colleague reports "our new filter is 90 percent accurate". What is the first question
you ask, and why? One or two sentences.

*Your explanation:* 